# Assignment Part 5 — Performance Testing (Load Testing)

Proves the system can handle OLTP-scale load:
1. **10,000 Create Case operations** — measures throughput and latency
2. **2,000 Create Appointment operations** across 8 rooms — measures concurrent booking throughput

In [7]:
import sys
sys.path.insert(0, '../src')

import time as time_module
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, time, timedelta
from rich.console import Console
from rich.table import Table
console = Console()

from sqlalchemy import select, text
from or_scheduler.database import engine, SessionLocal
from or_scheduler.models import Department, Staff, Room, Patient
from or_scheduler.operations import create_case, create_appointment, StaffItem, RoomConflictError

## Setup — Load References

In [8]:
from sqlalchemy.orm import Session

with Session(engine) as s:
    dept = s.execute(select(Department).limit(1)).scalar_one()
    surgeon = s.execute(select(Staff).where(Staff.role == 'SURGEON').limit(1)).scalar_one()
    anaest = s.execute(select(Staff).where(Staff.role == 'ANAESTHESIOLOGIST').limit(1)).scalar_one()
    all_surgeons = s.execute(select(Staff).where(Staff.role == 'SURGEON')).scalars().all()
    all_anaests  = s.execute(select(Staff).where(Staff.role == 'ANAESTHESIOLOGIST')).scalars().all()
    rooms = s.execute(select(Room).where(Room.room_type == 'OR')).scalars().all()
    patients = s.execute(select(Patient)).scalars().all()

    dept_id    = dept.department_id
    surgeon_id = surgeon.staff_id
    anaest_id  = anaest.staff_id
    surgeon_ids = [st.staff_id for st in all_surgeons]
    anaest_ids  = [an.staff_id for an in all_anaests]
    room_ids    = [r.room_id for r in rooms]
    patient_hns = [p.hn for p in patients]

print(f"Available rooms:              {len(room_ids)}")
print(f"Available patients:           {len(patient_hns)}")
print(f"Available surgeons:           {len(surgeon_ids)}")
print(f"Available anaesthesiologists: {len(anaest_ids)}")

Available rooms:              6
Available patients:           10000
Available surgeons:           5
Available anaesthesiologists: 5


## Test 1 — 10,000 Create Case Operations

Creates 10,000 surgical cases using batched transactions (100 per batch).
Pre-creates 10,000 patients to isolate Case insert performance.

In [13]:
def test_create_case_performance(n: int = 10_000) -> dict:
    """
    Insert n Cases in batches of 100 per transaction.
    Pre-seeds extra patients so each case has a valid patient.
    """
    BATCH_SIZE = 1
    
    # Pre-seed enough patients
    print(f"Pre-seeding {n} patients for performance test...")
    with SessionLocal() as seed_s:
        existing = seed_s.execute(text('SELECT COUNT(*) FROM patients')).scalar()
        needed = max(0, n - existing)
        if needed > 0:
            for i in range(needed):
                from or_scheduler.models import Patient
                import uuid
                # Bulk insert without checking duplicates for speed
                idx = existing + i + 1
                seed_s.add(Patient(
                    hn=f"PERF-{idx:08d}",
                    name=f"Perf Patient {idx}",
                    age=40
                ))
                if (i + 1) % 1000 == 0:
                    seed_s.commit()
                    print(f"  {i + 1}/{needed} patients seeded")
            seed_s.commit()
        print(f"  Patient pre-seeding complete.")
    
    # Load patient HNs
    with SessionLocal() as s:
        all_hns = [row[0] for row in s.execute(text('SELECT hn FROM patients LIMIT :n'), {'n': n})]
    
    latencies = []
    total_start = time_module.perf_counter()
    
    batches = [all_hns[i:i+BATCH_SIZE] for i in range(0, n, BATCH_SIZE)]
    
    for batch_idx, batch_hns in enumerate(batches):
        batch_start = time_module.perf_counter()
        with SessionLocal() as s:
            for hn in batch_hns:
                create_case(
                    s,
                    patient_hn=hn,
                    department_id=dept_id,
                    surgeon_id=surgeon_id,
                    procedure_type='Performance Test Procedure',
                    urgency='ELECTIVE',
                    created_by=surgeon_id,
                )
            s.commit()
        batch_end = time_module.perf_counter()
        latencies.append((batch_end - batch_start) * 1000 / BATCH_SIZE)  # ms per operation
        
        if (batch_idx + 1) % 20 == 0:
            print(f"  Batch {batch_idx + 1}/{len(batches)} complete")
    
    total_end = time_module.perf_counter()
    total_seconds = total_end - total_start
    tps = n / total_seconds
    
    return {
        'n': n,
        'total_seconds': total_seconds,
        'tps': tps,
        'p50_ms': statistics.median(latencies),
        'p95_ms': sorted(latencies)[int(len(latencies) * 0.95)],
        'p99_ms': sorted(latencies)[int(len(latencies) * 0.99)],
        'min_ms': min(latencies),
        'max_ms': max(latencies),
    }

print("Starting 10,000 create_case performance test...")
result1 = test_create_case_performance(10_000)
print("Done!")

Starting 10,000 create_case performance test...
Pre-seeding 10000 patients for performance test...
  Patient pre-seeding complete.
  Batch 20/10000 complete
  Batch 40/10000 complete
  Batch 60/10000 complete
  Batch 80/10000 complete
  Batch 100/10000 complete
  Batch 120/10000 complete
  Batch 140/10000 complete
  Batch 160/10000 complete
  Batch 180/10000 complete
  Batch 200/10000 complete
  Batch 220/10000 complete
  Batch 240/10000 complete
  Batch 260/10000 complete
  Batch 280/10000 complete
  Batch 300/10000 complete
  Batch 320/10000 complete
  Batch 340/10000 complete
  Batch 360/10000 complete
  Batch 380/10000 complete
  Batch 400/10000 complete
  Batch 420/10000 complete
  Batch 440/10000 complete
  Batch 460/10000 complete
  Batch 480/10000 complete
  Batch 500/10000 complete
  Batch 520/10000 complete
  Batch 540/10000 complete
  Batch 560/10000 complete
  Batch 580/10000 complete
  Batch 600/10000 complete
  Batch 620/10000 complete
  Batch 640/10000 complete
  Batch 6

In [14]:
t = Table(title="Performance Test 1: create_case() × 10,000", show_header=True, header_style="bold green")
t.add_column("Metric", style="cyan")
t.add_column("Value", style="yellow", justify="right")

t.add_row("Operations", f"{result1['n']:,}")
t.add_row("Total Time", f"{result1['total_seconds']:.2f} seconds")
t.add_row("Throughput", f"{result1['tps']:.0f} TPS")
t.add_row("P50 Latency", f"{result1['p50_ms']:.3f} ms")
t.add_row("P95 Latency", f"{result1['p95_ms']:.3f} ms")
t.add_row("P99 Latency", f"{result1['p99_ms']:.3f} ms")
t.add_row("Min Latency", f"{result1['min_ms']:.3f} ms")
t.add_row("Max Latency", f"{result1['max_ms']:.3f} ms")

console.print(t)

      Performance Test 1:      
    create_case() × 10,000     
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Metric      ┃         Value ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Operations  │        10,000 │
│ Total Time  │ 28.81 seconds │
│ Throughput  │       347 TPS │
│ P50 Latency │      2.258 ms │
│ P95 Latency │      5.472 ms │
│ P99 Latency │      9.688 ms │
│ Min Latency │      1.528 ms │
│ Max Latency │    213.041 ms │
└─────────────┴───────────────┘

## Test 2 — Concurrent Appointment Booking

Simulates 20 concurrent coordinators booking appointments across 5 OR rooms over a multi-day window. Each room is assigned a **unique surgeon + anaesthesiologist pair** (no staff shared across rooms) to measure pure booking throughput without staff conflicts. Uses `ThreadPoolExecutor` to run parallel booking attempts.

In [11]:
import uuid
from or_scheduler.models import RoomSchedule, StaffSchedule

def test_concurrent_booking_performance(n_bookings: int = 500, max_workers: int = 20) -> dict:
    """
    Simulate n_bookings concurrent appointment booking attempts across multiple rooms.
    Each OR room uses a unique surgeon + anaesthesiologist pair — no staff shared across rooms,
    so all conflicts are room-level only (not staff conflicts).
    """
    # One unique surgeon+anaest pair per room (5 pairs for 5 rooms)
    n_pairs  = min(len(surgeon_ids), len(anaest_ids))
    OR_rooms = room_ids[:min(n_pairs, len(room_ids))]

    # Performance test idempotency — clear previous run's reservations for the target date range.
    # Slots are deterministic (same date.today() formula each run), so prior committed reservations
    # would block every attempt on re-run, producing 500 conflicts and 0 successes.
    print("Clearing stale performance test reservations...")
    with SessionLocal() as _s:
        stale_ids = [str(r[0]) for r in _s.execute(text("""
            SELECT a.appointment_id FROM appointments a
            JOIN room_reservations rr ON rr.appointment_id = a.appointment_id
            WHERE rr.room_id::text = ANY(:rids)
              AND a.scheduled_date BETWEEN :d_min AND :d_max
        """), {
            'rids':  [str(r) for r in OR_rooms],
            'd_min': date.today() + timedelta(days=7),
            'd_max': date.today() + timedelta(days=35),
        }).fetchall()]

        if stale_ids:
            _s.execute(text("DELETE FROM equipment_reservations WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
            _s.execute(text("DELETE FROM staff_reservations     WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
            _s.execute(text("DELETE FROM room_reservations      WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
            _s.execute(text("DELETE FROM appointments           WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
            _s.commit()
            print(f"  Removed {len(stale_ids)} stale appointment(s) — clean slate")
        else:
            print("  No stale data — already clean")

    # Pre-create cases for each booking attempt
    print(f"Pre-creating {n_bookings} cases...")
    case_ids = []
    with SessionLocal() as s:
        all_hns = [row[0] for row in s.execute(text(f'SELECT hn FROM patients LIMIT {n_bookings}'))]

    with SessionLocal() as s:
        for hn in all_hns[:n_bookings]:
            cr = create_case(
                s, patient_hn=hn, department_id=dept_id,
                surgeon_id=surgeon_id, procedure_type='Perf Appointment Test',
                created_by=surgeon_id
            )
            case_ids.append(cr.case_id)
        s.commit()
    print(f"  {len(case_ids)} cases created.")

    # Build booking slots: each room gets unique time slots spread over multiple days.
    # Each room is assigned its own surgeon+anaest (room_idx → staff pair index).
    booking_slots = []
    for i, case_id in enumerate(case_ids):
        room_idx    = i % len(OR_rooms)
        day_offset  = (i // len(OR_rooms)) // 4 + 7  # start 7 days out
        slot_in_day = (i // len(OR_rooms)) % 4
        start_hour  = 8 + slot_in_day * 2
        if start_hour >= 17:
            continue
        booking_slots.append({
            'case_id':    case_id,
            'room_id':    OR_rooms[room_idx],
            'date':       date.today() + timedelta(days=day_offset),
            'start':      time(start_hour, 0),
            'end':        time(min(start_hour + 2, 17), 0),
            'surgeon_id': surgeon_ids[room_idx],   # unique per room — no staff conflict
            'anaest_id':  anaest_ids[room_idx],    # unique per room — no staff conflict
        })

    # Pre-create schedules for all future dates for all rooms and all staff pairs
    extra_dates = set(slot['date'] for slot in booking_slots)
    with SessionLocal() as s:
        for d in extra_dates:
            for room_id in OR_rooms:
                existing = s.execute(
                    select(RoomSchedule).where(RoomSchedule.room_id == room_id, RoomSchedule.date == d)
                ).scalar_one_or_none()
                if not existing:
                    s.add(RoomSchedule(room_id=room_id, date=d,
                                       available_from=time(8, 0), available_until=time(17, 0),
                                       schedule_type='REGULAR'))
            for staff_id in surgeon_ids + anaest_ids:
                existing = s.execute(
                    select(StaffSchedule).where(
                        StaffSchedule.staff_id == staff_id, StaffSchedule.date == d
                    )
                ).scalar_one_or_none()
                if not existing:
                    s.add(StaffSchedule(staff_id=staff_id, date=d,
                                        available_from=time(8, 0), available_until=time(17, 0),
                                        schedule_type='REGULAR'))
        s.commit()

    print(f"Booking {len(booking_slots)} appointments with {max_workers} workers across {len(OR_rooms)} rooms...")

    successes = 0
    conflicts = 0
    errors    = 0
    latencies = []

    def book_one(slot):
        start = time_module.perf_counter()
        try:
            with SessionLocal() as s:
                create_appointment(
                    s,
                    case_id=slot['case_id'],
                    room_id=slot['room_id'],
                    scheduled_date=slot['date'],
                    start_time=slot['start'],
                    end_time=slot['end'],
                    staff_items=[
                        StaffItem(slot['surgeon_id'], 'SURGEON'),
                        StaffItem(slot['anaest_id'],  'ANAESTHESIOLOGIST'),
                    ],
                    confirmed_by=slot['surgeon_id'],
                )
                s.commit()
            return ('success', (time_module.perf_counter() - start) * 1000)
        except RoomConflictError:
            return ('conflict', (time_module.perf_counter() - start) * 1000)
        except Exception as e:
            return ('error', str(e))

    total_start = time_module.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(book_one, slot) for slot in booking_slots]
        for future in as_completed(futures):
            result_type, val = future.result()
            if result_type == 'success':
                successes += 1
                latencies.append(val)
            elif result_type == 'conflict':
                conflicts += 1
            else:
                errors += 1
                print(f"  Unexpected error: {val}")
    total_end = time_module.perf_counter()

    total_seconds = total_end - total_start

    return {
        'n_rooms':       len(OR_rooms),
        'n_attempted':   len(booking_slots),
        'successes':     successes,
        'conflicts':     conflicts,
        'errors':        errors,
        'total_seconds': total_seconds,
        'tps':           successes / total_seconds,
        'p50_ms': statistics.median(latencies) if latencies else 0,
        'p95_ms': sorted(latencies)[int(len(latencies) * 0.95)] if latencies else 0,
        'p99_ms': sorted(latencies)[int(len(latencies) * 0.99)] if latencies else 0,
    }

result2 = test_concurrent_booking_performance(500)
print("Done!")

Clearing stale performance test reservations...
  No stale data — already clean
Pre-creating 500 cases...
  500 cases created.
Booking 500 appointments with 20 workers across 5 rooms...
Done!


In [12]:
t = Table(title="Performance Test 2: create_appointment() — Concurrent Booking", show_header=True, header_style="bold green")
t.add_column("Metric", style="cyan")
t.add_column("Value", style="yellow", justify="right")

t.add_row("OR Rooms used",           f"{result2['n_rooms']}")
t.add_row("Attempts",                f"{result2['n_attempted']:,}")
t.add_row("Successes",               f"{result2['successes']:,}") 
t.add_row("Conflicts (room)",        f"{result2['conflicts']:,}")
t.add_row("Errors (unexpected)",     f"{result2['errors']:,}")
t.add_row("Total Time",              f"{result2['total_seconds']:.2f} seconds")
t.add_row("Throughput (successful)", f"{result2['tps']:.0f} TPS")
t.add_row("P50 Latency",             f"{result2['p50_ms']:.1f} ms")
t.add_row("P95 Latency",             f"{result2['p95_ms']:.1f} ms")
t.add_row("P99 Latency",             f"{result2['p99_ms']:.1f} ms")

console.print(t)

print("\nConclusion:")
print(f"  {result2['successes']} appointments committed across {result2['n_rooms']} OR rooms with unique staff pairs.")
print(f"  {result2['conflicts']} room conflicts detected (expected: ~0 — each room has unique non-overlapping slots).")
print(f"  {result2['errors']} unexpected errors (expected: 0).")

Performance Test 2: create_appointment() —
            Concurrent Booking            
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Metric                  ┃        Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ OR Rooms used           │            5 │
│ Attempts                │          500 │
│ Successes               │          500 │
│ Conflicts (room)        │            0 │
│ Errors (unexpected)     │            0 │
│ Total Time              │ 2.74 seconds │
│ Throughput (successful) │      183 TPS │
│ P50 Latency             │     101.5 ms │
│ P95 Latency             │     188.9 ms │
│ P99 Latency             │     213.5 ms │
└─────────────────────────┴──────────────┘


Conclusion:
  500 appointments committed across 5 OR rooms with unique staff pairs.
  0 room conflicts detected (expected: ~0 — each room has unique non-overlapping slots).
  0 unexpected errors (expected: 0).
